# Notebook 05 — Grad-CAM and Ablation Study
**Grad-CAM:** Spatial explainability — which regions drive the classification.
**Ablation:** Compare three model variants to justify design choices.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import json
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from src.config import (
    MODELS_DIR, PLOTS_DIR, RESULTS_DIR, SEED, CLASSES,
    ANOMALY_CLASSES, CLASS_TO_IDX, CLIP_LEN, JOINT_EPOCHS,
    LEARNING_RATE, BATCH_SIZE, NUM_CLASSES
)
from src.model import (
    Encoder, Decoder, LSTMHead, JointModel,
    SingleFrameCNN, freeze_decoder
)
from src.dataset import load_npy_split, load_normal_npy, make_loader
from src.gradcam import compute_gradcam, overlay_heatmap, visualize_gradcam_batch
from src.evaluate import (
    compute_reconstruction_errors, find_optimal_threshold, evaluate_classifier
)
from src.train import pretrain_autoencoder, joint_train

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"PyTorch: {torch.__version__}  Device: {DEVICE}")

## 1. Load Data and Best Joint Model

In [ ]:
X_test, y_test = load_npy_split("test")
X_val,  y_val  = load_npy_split("val")
X_normal       = load_normal_npy()
X_train, y_train = load_npy_split("train")

encoder = Encoder(); decoder = Decoder(); lstm_head = LSTMHead()
freeze_decoder(decoder)
joint_model = JointModel(encoder, decoder, lstm_head)
joint_model.load_state_dict(
    torch.load(MODELS_DIR / "joint_model_best.pth", map_location=DEVICE))
joint_model = joint_model.to(DEVICE).eval()
print("Models loaded.")

## 2. Grad-CAM

### 2a. Single Example

In [ ]:
fight_idx = np.where(y_test == CLASS_TO_IDX["Fighting"])[0][0]
clip      = X_test[fight_idx]

heatmap, pred_cls, conf = compute_gradcam(
    encoder.to(DEVICE), lstm_head.to(DEVICE),
    clip, class_idx=CLASS_TO_IDX["Fighting"],
    frame_idx=CLIP_LEN // 2, device=DEVICE
)

frame   = clip[CLIP_LEN // 2]
overlay = overlay_heatmap(frame, heatmap, alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(frame);               axes[0].set_title("Input Frame");    axes[0].axis("off")
axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Grad-CAM");       axes[1].axis("off")
axes[2].imshow(overlay);             axes[2].axis("off")
axes[2].set_title(f"Overlay\nPred: {pred_cls} ({conf:.2f})")
plt.suptitle("Grad-CAM Example: Fighting", fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "gradcam_example.png", dpi=150, bbox_inches="tight")
plt.show()

### 2b. Batch Visualisation (2 examples per anomaly class)

In [ ]:
visualize_gradcam_batch(
    encoder.to(DEVICE), lstm_head.to(DEVICE),
    X_test, y_test, n_per_class=2,
    save_path=PLOTS_DIR / "gradcam_batch.png", device=DEVICE
)
plt.show()

## 3. Ablation Study

### Variant A: Single-frame CNN (no LSTM, no temporal context)

In [ ]:
print("=" * 55)
print("VARIANT A: Single-frame CNN (no LSTM)")
print("=" * 55)

mid    = CLIP_LEN // 2
# Extract middle frame: (N, H, W, C) -> (N, C, H, W) tensor
def _frames(X):
    t = torch.from_numpy(X[:, mid].transpose(0, 3, 1, 2)).float()
    return t

sf_model   = SingleFrameCNN().to(DEVICE)
optimizer  = torch.optim.Adam(sf_model.parameters(), lr=LEARNING_RATE)
ce_loss    = nn.CrossEntropyLoss()

from torch.utils.data import TensorDataset, DataLoader
tr_ds = DataLoader(TensorDataset(_frames(X_train), torch.from_numpy(y_train).long()),
                   batch_size=BATCH_SIZE, shuffle=True)
va_ds = DataLoader(TensorDataset(_frames(X_val),   torch.from_numpy(y_val).long()),
                   batch_size=BATCH_SIZE)

best_sf_val = float("inf"); sf_patience = 0
for epoch in range(JOINT_EPOCHS):
    sf_model.train()
    for frames, labels in tr_ds:
        frames, labels = frames.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        ce_loss(sf_model(frames), labels).backward()
        optimizer.step()
    sf_model.eval()
    va_loss = 0
    with torch.no_grad():
        for frames, labels in va_ds:
            frames, labels = frames.to(DEVICE), labels.to(DEVICE)
            va_loss += ce_loss(sf_model(frames), labels).item()
    if va_loss < best_sf_val:
        best_sf_val = va_loss; sf_patience = 0
        torch.save(sf_model.state_dict(), MODELS_DIR / "ablation_sf.pth")
    else:
        sf_patience += 1
        if sf_patience >= 5: break

sf_model.load_state_dict(torch.load(MODELS_DIR / "ablation_sf.pth", map_location=DEVICE))
sf_model.eval()
te_frames = _frames(X_test).to(DEVICE)
with torch.no_grad():
    sf_preds = sf_model(te_frames).argmax(dim=1).cpu().numpy()

from sklearn.metrics import f1_score, classification_report as cr
sf_f1 = f1_score(y_test, sf_preds, average="macro", zero_division=0)
print(f"\nVariant A: Macro F1: {sf_f1:.4f}")
print(cr(y_test, sf_preds, target_names=CLASSES, zero_division=0))

### Variant B: CNN+LSTM, UNFROZEN decoder

In [ ]:
print("=" * 55)
print("VARIANT B: CNN+LSTM with UNFROZEN decoder")
print("=" * 55)

from src.model import Autoencoder as AE

enc_b = Encoder(); dec_b = Decoder(); lstm_b = LSTMHead()
ae_b  = AE(enc_b, dec_b)
pretrain_autoencoder(ae_b, X_normal[:5000], val_split=0.1, device=DEVICE)

joint_b = JointModel(enc_b, dec_b, lstm_b)   # decoder NOT frozen
hist_b  = joint_train(joint_b, X_train, y_train, X_val, y_val, device=DEVICE)
torch.save(joint_b.state_dict(), MODELS_DIR / "ablation_unfrozen.pth")

y_pred_b, report_b = evaluate_classifier(joint_b.to(DEVICE), X_test, y_test, device=DEVICE)
b_f1 = report_b["macro avg"]["f1-score"]

b_errors = compute_reconstruction_errors(joint_b, X_test, device=DEVICE)
b_normal = compute_reconstruction_errors(joint_b, X_val[y_val == CLASS_TO_IDX["Normal"]], device=DEVICE)
print(f"\nVariant B: Macro F1: {b_f1:.4f}")
print(f"Normal MSE mean  : {b_normal.mean():.5f}")
print(f"Anomaly MSE mean : {b_errors[y_test!=0].mean():.5f}")
print("(Gap should be small — unfrozen decoder learns anomalies too)")

### Variant C: CNN+LSTM, FROZEN decoder (our model)

In [ ]:
print("=" * 55)
print("VARIANT C: CNN+LSTM, FROZEN decoder (our model)")
print("=" * 55)

y_pred_c, report_c = evaluate_classifier(joint_model, X_test, y_test, device=DEVICE)
c_f1 = report_c["macro avg"]["f1-score"]
c_errors = compute_reconstruction_errors(joint_model, X_test, device=DEVICE)
c_normal = compute_reconstruction_errors(joint_model, X_val[y_val==CLASS_TO_IDX["Normal"]], device=DEVICE)

from sklearn.metrics import roc_auc_score
c_auc = roc_auc_score((y_test != 0).astype(int), c_errors)
b_auc = roc_auc_score((y_test != 0).astype(int), b_errors)

print(f"\nVariant C: Macro F1: {c_f1:.4f}  Anomaly AUC: {c_auc:.4f}")
print(f"Normal MSE mean  : {c_normal.mean():.5f}")
print(f"Anomaly MSE mean : {c_errors[y_test!=0].mean():.5f}")

### Ablation Summary Table

In [ ]:
ablation = {
    "A: Single-frame CNN (no LSTM)": {
        "Anomaly AUC": "N/A", "Classification F1": round(sf_f1, 4), "Notes": "No temporal context"},
    "B: CNN+LSTM, unfrozen decoder": {
        "Anomaly AUC": round(b_auc, 4), "Classification F1": round(b_f1, 4),
        "Notes": "Decoder learns anomalies, weak anomaly signal"},
    "C: CNN+LSTM, frozen decoder (ours)": {
        "Anomaly AUC": round(c_auc, 4), "Classification F1": round(c_f1, 4),
        "Notes": "Best overall performance"},
}

print("\n" + "=" * 75)
print(f"{'Model Variant':<38} {'AUC':>8} {'F1':>8}  Notes")
print("-" * 75)
for name, vals in ablation.items():
    print(f"  {name:<36} {str(vals['Anomaly AUC']):>8} {vals['Classification F1']:>8.4f}  {vals['Notes']}")
print("=" * 75)

with open(RESULTS_DIR / "ablation_results.json", "w") as f:
    json.dump(ablation, f, indent=2, default=str)

### Ablation Bar Chart

In [ ]:
variants  = ["A: Single-frame\nCNN", "B: CNN+LSTM\nUnfrozen", "C: CNN+LSTM\nFrozen (ours)"]
f1_scores = [sf_f1, b_f1, c_f1]
colors    = ["#AAAAAA", "#888888", "#222222"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(variants, f1_scores, color=colors, edgecolor="black", linewidth=0.8)
ax.set_ylim(0, 1.1); ax.set_ylabel("Macro F1 Score")
ax.set_title("Ablation Study: Classification F1 by Model Variant", fontweight="bold")
for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{score:.4f}", ha="center", fontsize=10, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ablation_f1.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n[DONE] Grad-CAM and Ablation complete.")
print(f"Results: {RESULTS_DIR}  |  Plots: {PLOTS_DIR}")